# Fine-Tuning an LLM for Text-to-SQL with QLoRA (Colab Pro)

End-to-end: install -> inspect data -> train (QLoRA SFT) -> evaluate (base vs tuned) ->
infer -> merge & push.

**Before you start:** `Runtime -> Change runtime type -> GPU`. A **T4 (16 GB)** is enough
for the 1.5B default. Run cells top to bottom.

## 1. Check the GPU you got

In [ ]:
!nvidia-smi

## 2. Install the stack

Pinned to the **current TRL API** (`processing_class`, `SFTConfig.max_length`). If you
already cloned the repo into Colab you can `pip install -r requirements.txt` instead.

In [ ]:
!pip install -q -U \
  "transformers>=4.46.0" "trl>=0.13.0" "peft>=0.13.0" \
  "bitsandbytes>=0.44.0" "accelerate>=1.1.0" "datasets>=3.0.0" \
  "sqlglot>=25.0.0" "pydantic>=2.0.0" "pyyaml>=6.0"
print("Installed. If you hit an import error, Runtime -> Restart session, then re-run.")

## 3. Get the project code

Two options. **A)** clone your GitHub repo. **B)** if you're prototyping in a bare Colab,
run the next cell to write the minimal modules inline. (Option A is recommended once your
repo is up.)

In [ ]:
# Option A: clone your repo (edit the URL), then skip the inline-write cell.
# !git clone https://github.com/<you>/llm-finetuning-text2sql.git
# %cd llm-finetuning-text2sql
print("If cloning, %cd into the repo and skip Option B below.")

### Option B (self-contained): minimal inline modules
Run this only if you did NOT clone the repo.

In [ ]:
import os, textwrap, pathlib
# Mirrors src/data.py — keep this identical to the repo to avoid train/serve drift.
SYSTEM_PROMPT = ("You are a precise text-to-SQL assistant. Given a database schema and a "
                 "question, respond with a single valid SQL query and nothing else.")

def build_user_prompt(schema, question):
    return (f"### Database schema:\n{schema.strip()}\n\n"
            f"### Question:\n{question.strip()}\n\n### SQL:")
print("Inline helpers ready.")

## 4. Config (edit here to scale up on L4/A100)

In [ ]:
MODEL_ID   = "Qwen/Qwen2.5-1.5B-Instruct"   # L4/A100: "Qwen/Qwen2.5-3B-Instruct" / "...-7B-Instruct"
DATASET    = "b-mc2/sql-create-context"
OUT_DIR    = "outputs/qwen2.5-1.5b-text2sql-qlora"
MAX_TRAIN  = 8000     # set None for full dataset (~78k); start small to validate the loop
MAX_EVAL   = 500
MAX_LEN    = 1024
BATCH      = 2        # L4: 4, A100: 8
GRAD_ACC   = 8        # effective batch = BATCH * GRAD_ACC
EPOCHS     = 1
LR         = 2e-4

## 5. Load + format data (conversational `messages`)

In [ ]:
from datasets import load_dataset

raw = load_dataset(DATASET, split="train")
split = raw.train_test_split(test_size=0.05, seed=42)

def to_row(ex):
    return {
        "schema": ex["context"], "question": ex["question"], "gold_sql": ex["answer"].strip(),
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(ex["context"], ex["question"])},
            {"role": "assistant", "content": ex["answer"].strip()},
        ],
    }

train_ds = split["train"].select(range(min(MAX_TRAIN, len(split["train"])))).map(to_row)
eval_ds  = split["test"].select(range(min(MAX_EVAL,  len(split["test"])))).map(to_row)
print(f"train={len(train_ds)}  eval={len(eval_ds)}")

### Sanity-check: render one example with the chat template
The SQL must appear in the assistant turn. **Never train without eyeballing this.**

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(tokenizer.apply_chat_template(train_ds[0]["messages"], tokenize=False))

## 6. Load base model in 4-bit (QLoRA) + prep for k-bit training

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto", attn_implementation="eager",
)
model.config.use_cache = False  # required with gradient checkpointing
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
print("Base loaded in 4-bit and prepared.")

## 7. Train (LoRA adapters via TRL `SFTTrainer`)

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

sft_config = SFTConfig(
    output_dir=OUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LR, lr_scheduler_type="cosine", warmup_ratio=0.03,
    max_length=MAX_LEN,                       # NOTE: not max_seq_length
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit", bf16=True,
    logging_steps=10, eval_strategy="steps", eval_steps=100,
    save_strategy="steps", save_steps=200, save_total_limit=2,
    report_to="tensorboard", seed=42,
)

trainer = SFTTrainer(
    model=model, args=sft_config,
    train_dataset=train_ds, eval_dataset=eval_ds,
    peft_config=peft_config,
    processing_class=tokenizer,               # NOTE: not tokenizer=
)
trainer.model.print_trainable_parameters()    # confirm <1% trainable
trainer.train()
trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print("Done. Adapter saved to", OUT_DIR)

### (optional) View training curves

In [ ]:
%load_ext tensorboard
%tensorboard --logdir outputs

## 8. Evaluate — base vs fine-tuned

We measure **AST-match** (structural SQL equivalence via `sqlglot`), exact-match, and
parse-rate. Run the eval helper once, then call it on each model.

In [ ]:
import re, sqlglot

def _gen(m, q_schema, q_text, max_new=256):
    msgs = [{"role":"system","content":SYSTEM_PROMPT},
            {"role":"user","content":build_user_prompt(q_schema, q_text)}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ins = tokenizer(prompt, return_tensors="pt").to(m.device)
    with torch.inference_mode():
        out = m.generate(**ins, max_new_tokens=max_new, do_sample=False,
                         eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][ins["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def _norm(s): return re.sub(r"\s+"," ", s.strip().rstrip(";").strip()).lower()
def _parses(s):
    try: sqlglot.parse_one(s); return True
    except Exception: return False
def _ast_eq(g,p):
    try:
        return (sqlglot.parse_one(g).sql(normalize=True).lower()
                == sqlglot.parse_one(p).sql(normalize=True).lower())
    except Exception: return False

def evaluate(m, ds, limit=200):
    n=parse=exact=ast=0
    for row in ds.select(range(min(limit,len(ds)))):
        pred=_gen(m,row["schema"],row["question"]); gold=row["gold_sql"]; n+=1
        parse+=_parses(pred); exact+=(_norm(pred)==_norm(gold)); ast+=_ast_eq(gold,pred)
    return {"n":n,"parse_rate":parse/n,"exact_match":exact/n,"ast_match":ast/n}

# Fine-tuned model is the currently-loaded `model` (base + adapter in memory after training)
tuned_scores = evaluate(model, eval_ds, limit=200)
print("FINE-TUNED:", tuned_scores)

### Baseline: reload the un-tuned base and score it for comparison

In [ ]:
base_only = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto", attn_implementation="eager")
base_only.eval()
base_scores = evaluate(base_only, eval_ds, limit=200)
print("BASE     :", base_scores)
print("FINE-TUNED:", tuned_scores)
del base_only; torch.cuda.empty_cache()

## 9. Try your model on a new question

In [ ]:
schema = "CREATE TABLE head (age INTEGER, name TEXT)"
question = "How many heads are older than 56?"
print(_gen(model, schema, question))

## 10. (optional) Merge adapter -> standalone model and push to Hub

Merging must be done in bf16 (not 4-bit). Then publish with a model card.

In [ ]:
# from huggingface_hub import notebook_login; notebook_login()
# from peft import PeftModel
# base_bf16 = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")
# merged = PeftModel.from_pretrained(base_bf16, OUT_DIR).merge_and_unload()
# merged.save_pretrained("outputs/merged", safe_serialization=True)
# tokenizer.save_pretrained("outputs/merged")
# merged.push_to_hub("<you>/qwen2.5-1.5b-text2sql"); tokenizer.push_to_hub("<you>/qwen2.5-1.5b-text2sql")
print("Uncomment to merge + push once you have a HF token.")

---
### What to commit to GitHub after this notebook
- This notebook (with outputs cleared: `Edit -> Clear all outputs`).
- Your filled-in results table in `README.md` (base vs fine-tuned).
- Push the model/adapter to the Hub and link it in the README.

See `CURRICULUM.md` for the full 6-week plan and interview prep.